# Intercropping parentage likelihood: pixels → fields → four windows

This notebook implements the DNA-inspired analysis as two separate questions:

1. **Evidence:** does a named crop pair explain a field better than either pure parent or another crop pair?
2. **Attribution:** conditional on that model, what is the balance of parent-A-like and parent-B-like pixel signatures?

The 128 TESSERA dimensions are fitted jointly with a field-balanced, shrunken covariance model. Monocrop field labels provide weak supervision for their pixels; intercropping labels are never treated as per-pixel truth. Validation holds out complete physical fields. Pixel maps show the actual 10 m WKT-selected cells; field estimates use a mosaic likelihood and 20 m block bootstrap. `w1`–`w4` are evaluated separately because the cumulative windows are not independent.

The notebook previews only a few deterministic fields, then saves one four-window dashboard for **every canonical physical field complete in all four windows**. Exact same-label WKT replicas share that physical report, while `source_field_scores.parquet` and `field_plot_index.parquet` retain every original `field_uid`. Conflicting-label geometries remain explicit no-calls.

> Shares are relative **embedding-signature shares**. They are not planted-area, plant-count, biomass, yield, or sub-pixel abundance percentages without known-ratio calibration data.


In [ ]:
from dataclasses import replace
from pathlib import Path
import os
import sys

from IPython import get_ipython
from IPython.display import Markdown, display

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "plain_tessera_incremental" / "__init__.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the SpectraJam checkout.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from plain_tessera_incremental.notebooks._dna_reporting import (
    plot_cohort_overview,
    plot_field_dashboard,
    save_all_field_reports,
)
from plain_tessera_incremental.notebooks._dna_workflow import (
    default_config,
    finalize_workflow_export,
    run_workflow,
    save_field_plot_index,
    save_workflow_tables,
)

OUTPUT_DIR = Path(
    os.environ.get(
        "TESSERA_OUTPUT_DIR",
        "/mnt/noobjam/harvard_tessera_incremental_v2",
    )
)
EXPORT_BASE = Path(
    os.environ.get(
        "TESSERA_DNA_EXPORT_DIR",
        str(OUTPUT_DIR / "analysis" / "intercropping_parentage_likelihood_v1"),
    )
)
TEST_MODE = os.environ.get("TESSERA_DNA_TEST_MODE", "0") == "1"
REPORT_LIMIT = int(os.environ.get("TESSERA_DNA_REPORT_LIMIT", "0"))
PREVIEW_FIELDS = int(os.environ.get("TESSERA_DNA_PREVIEW_FIELDS", "4"))

config = default_config(OUTPUT_DIR, EXPORT_BASE)
if TEST_MODE:
    config = replace(
        config,
        shrinkage_candidates=(0.5,),
        max_folds=2,
        bootstrap_replicates=8,
        mixture_grid_size=41,
        min_reference_fields_per_crop=2,
        min_balanced_accuracy=0.0,
        max_endpoint_mae=1.0,
        max_synthetic_mae=1.0,
    )

print("Pipeline output:", OUTPUT_DIR)
print("Analysis export base:", EXPORT_BASE)
print("Report policy:", "ALL complete physical fields" if REPORT_LIMIT == 0 else f"first {REPORT_LIMIT}")


## 1. Freeze the published shards and run the numerical analysis

The source loader validates run fingerprints, task membership, deterministic row IDs, schemas, and finite `float32[128]` embeddings. The workflow scores every currently published canonical physical field and later maps results back to every source row. No 5 km grid filters or subsamples the data; validation holds out entire fields and is not presented as geographic generalization.


In [ ]:
result = run_workflow(config, progress=print)

source_accounting = (
    result.field_crosswalk.groupby("record_role")
    .agg(
        source_rows=("field_uid", "size"),
        physical_units=("analysis_unit_uid", "nunique"),
    )
    .sort_values("source_rows", ascending=False)
)
display(Markdown("### Source-field accounting"))
display(source_accounting)
display(Markdown("### Available/scored data by window"))
display(result.window_summary.set_index("window_id"))
display(Markdown("### Held-out reference validation"))
validation_summary = (
    result.validation_metrics.drop_duplicates(["window_id", "pair_key"])
    [[
        "window_id",
        "pair_key",
        "reference_fields_parent_a",
        "reference_fields_parent_b",
        "field_balanced_accuracy",
        "endpoint_mae",
        "synthetic_mosaic_mae",
        "validation_gate_passed",
    ]]
)
display(validation_summary.set_index(["pair_key", "window_id"]))


## 2. Save every numerical result first

Tables are written before rendering the large field gallery. If figure generation is interrupted, the numerical snapshot remains complete and can be reused. The snapshot path includes both the pipeline fingerprint and the exact published-shard fingerprint, so reruns cannot silently mix states.


In [ ]:
export_root, analysis_manifest = save_workflow_tables(result)
print("Frozen export root:", export_root)
print("Source field × window rows:", f"{len(result.source_field_scores):,}")
print("Canonical pixel × window rows:", f"{len(result.pixel_scores):,}")
display(
    result.source_field_scores[
        [
            "field_uid",
            "analysis_unit_uid",
            "window_id",
            "landcover_source",
            "mosaic_parent_a_share",
            "log_evidence_over_pure",
            "named_parent_mass",
            "call_status",
        ]
    ].head(12)
)


## 3. Define the honest four-window report cohort

A four-window dashboard is created only when the same physical field has pixel evidence in all four windows. Fields that are pending, empty, unsupported, or incomplete remain present in the source tables and plot index with an explicit status.


In [ ]:
complete_window_counts = (
    result.physical_field_pair_scores.groupby("field_uid")["window_id"]
    .nunique()
)
all_report_field_ids = complete_window_counts[
    complete_window_counts.eq(len(config.windows))
].index.astype(str).tolist()
report_field_ids = all_report_field_ids.copy()
if REPORT_LIMIT > 0:
    report_field_ids = report_field_ids[:REPORT_LIMIT]
gallery_complete = len(report_field_ids) == len(all_report_field_ids)
if not report_field_ids:
    raise RuntimeError("No physical field is scoreable in all four windows yet.")

report_field_evidence = result.physical_field_pair_scores[
    result.physical_field_pair_scores["field_uid"].isin(report_field_ids)
].copy()
report_pixel_evidence = result.pixel_scores[
    result.pixel_scores["field_uid"].isin(report_field_ids)
].copy()
report_fields = result.analysis_units[
    result.analysis_units["analysis_unit_uid"].isin(report_field_ids)
]["field_uid wkt landcover source_replica_count geometry_label_count".split()].copy()

print(f"Four-window gallery: {len(report_field_ids):,}/{len(all_report_field_ids):,} unique physical fields")
print("Gallery complete:", gallery_complete)
print("Original source rows represented by those reports:", f"{result.field_crosswalk['analysis_unit_uid'].isin(report_field_ids).sum():,}")
display(
    report_fields.groupby("landcover")
    .agg(physical_fields=("field_uid", "nunique"), source_rows=("source_replica_count", "sum"))
    .sort_values("physical_fields", ascending=False)
)


## 4. Cohort-level evidence overview

These panels show denominators, evidence relative to pure-parent controls, intercropping balance through the cumulative windows, and every call status. PCA is deliberately not used to manufacture percentages.


In [ ]:
overview = plot_cohort_overview(report_field_evidence, windows=config.windows)
display(overview)
plt.close(overview)


## 5. Preview a few meaningful fields

Examples are deterministic: for each intercropping label, the largest field and the field closest to a 50:50 `w4` mosaic estimate are preferred. Each dashboard uses one common parent scale across `w1`–`w4`, shows actual pixels and WKT, block-bootstrap uncertainty, evidence versus the pure-parent threshold, all-crop probability mass, and the best alternative pair.


In [ ]:
w4 = report_field_evidence[report_field_evidence["window_id"].eq("w4")].copy()
label_pair = dict(config.mixture_labels)
preview_ids = []
for label, pair_key in label_pair.items():
    candidates = w4[
        w4["landcover"].eq(label) & w4["pair_key"].eq(pair_key)
    ].copy()
    if candidates.empty:
        continue
    largest = candidates.sort_values(
        ["pixel_count", "field_uid"], ascending=[False, True]
    ).iloc[0]["field_uid"]
    balanced = (
        candidates.assign(distance_to_half=lambda frame: (frame["mosaic_parent_a_share"] - 0.5).abs())
        .sort_values(["distance_to_half", "pixel_count", "field_uid"], ascending=[True, False, True])
        .iloc[0]["field_uid"]
    )
    preview_ids.extend([str(largest), str(balanced)])
preview_ids = list(dict.fromkeys(preview_ids))[:PREVIEW_FIELDS]
if not preview_ids:
    preview_ids = report_field_ids[:PREVIEW_FIELDS]

for field_uid in preview_ids:
    figure = plot_field_dashboard(
        field_uid,
        report_pixel_evidence,
        report_field_evidence,
        report_fields,
        windows=config.windows,
    )
    display(figure)
    plt.close(figure)


## 6. Save the complete field gallery

This is the long-running cell. It renders one dashboard per unique complete physical field, then writes a SHA-256 plot manifest. Same-WKT replicas point to the same dashboard instead of generating duplicate images. Progress is printed every 100 reports. A deliberate `TESSERA_DNA_REPORT_LIMIT` smoke run writes `SMOKE_COMPLETE.json`, never the full `COMPLETED.json`.


In [ ]:
plot_output = save_all_field_reports(
    report_pixel_evidence,
    report_field_evidence,
    report_fields,
    export_root / "figures",
    windows=config.windows,
    dpi=110 if TEST_MODE else 170,
    metadata={
        "run_fingerprint": result.static.run_fingerprint,
        "snapshot_id": result.snapshot_id,
        "analysis_id": result.analysis_id,
        "scientific_unit": "unique physical geometry+label field",
    },
    progress=print,
)
plot_index = save_field_plot_index(
    result,
    export_root,
    plot_output["field_paths"],
    report_limited=not gallery_complete,
)
finalize_workflow_export(
    export_root,
    analysis_manifest,
    field_plot_count=len(plot_output["field_paths"]),
    plot_manifest_path=str(Path(plot_output["manifest_path"]).resolve().relative_to(export_root.resolve())),
    gallery_complete=gallery_complete,
    report_limit=REPORT_LIMIT if REPORT_LIMIT > 0 else None,
)

print("Overview:", plot_output["overview_path"])
print("Plot manifest:", plot_output["manifest_path"])
print("Reports saved:", f"{len(plot_output['field_paths']):,}")
display(plot_index["report_status"].value_counts().to_frame("source_fields"))


## 7. Completion audit

`COMPLETED.json` is written only after all numerical tables, the overview, every complete physical-field report, the plot manifest, and the all-source plot index have been published. A limited smoke gallery is explicitly incomplete.


In [ ]:
expected_source_rows = len(result.field_crosswalk) * len(config.windows)
assert len(result.source_field_scores) == expected_source_rows
assert not result.source_field_scores.duplicated(["field_uid", "window_id"]).any()
assert len(plot_index) == len(result.field_crosswalk)
completion_marker = export_root / ("COMPLETED.json" if gallery_complete else "SMOKE_COMPLETE.json")
assert completion_marker.is_file()
if not gallery_complete:
    assert not (export_root / "COMPLETED.json").exists()

display(
    pd.DataFrame(
        {
            "artifact": [
                "analysis manifest",
                "all source field/window scores",
                "all-source plot index",
                "figure manifest",
                "completion marker",
            ],
            "path": [
                export_root / "analysis_manifest.json",
                export_root / "tables" / "source_field_scores.parquet",
                export_root / "tables" / "field_plot_index.parquet",
                Path(plot_output["manifest_path"]),
                completion_marker,
            ],
        }
    )
)
print("Analysis complete." if gallery_complete else "Smoke gallery complete; full gallery is not marked complete.")


## Interpretation checklist

- `INTERCROP_SUPPORTED` requires the field-held-out validation gate, evidence above the empirical pure-parent threshold, adequate named-parent mass, an interior mosaic share and block-bootstrap interval for labelled mixtures, and no better alternative pair.
- `PARENT_A_LIKE` / `PARENT_B_LIKE` means the field behaves like an endpoint under that named-pair model.
- `evidence_tail_probability` is a held-out-reference tail diagnostic, not a formal exchangeable p-value; target pixels are spatially dependent and the final model uses all references.
- `OUT_OF_FAMILY` means another crop pair or off-pair probability explains the field better.
- `NO_CALL_ONE_PIXEL` is intentional: one pixel cannot establish a spatial two-contributor mosaic.
- The block-bootstrap interval measures within-field spatial stability; it does not convert the signature share into a physical planting ratio.
- `w4` remains an out-of-contract 487-day sensitivity window and should be marked as such in scientific reporting.
